### **FIMbench - `publish`**

The shared module that **pushes** FIMbench content out to its destinations: a
benchmark flood map standardized by `processing_floodmap`, and the catalog core +
vector tiles built by `webcontent_utils` - to the S3 database and to ArcGIS Online.

### **Install using PyPI**

In [ ]:
!uv pip install fimbench

### **AWS credentials**

All S3 publishing goes through one access layer, so credentials are **optional**
and resolve as: explicit keys -> device credentials -> interactive prompt. The
usual call passes **no** credentials and relies on the device.

### **Scenario 1 - publish a standardized flood map folder to S3**

The folder is the per-map output of `processing_floodmap`. Every keyword is shown.

In [ ]:
from fimbench import upload_benchmarkfloodmap

upload_benchmarkfloodmap(
    'out/AI_..._folder/',         # local file or folder to upload (recursive)
    bucket='sdmlab',              # destination S3 bucket
    prefix='FIM_Database/',       # destination key prefix
    aws_access_key_id=None,       # explicit key (None -> device creds / prompt)
    aws_secret_access_key=None,   # explicit secret (None -> device creds / prompt)
    region=None,                  # AWS region, e.g. 'us-east-1'
    profile=None,                 # named AWS profile (optional)
)

### **Scenario 2 - push the catalog core + tiles to S3**

Takes the catalog core + vector tiles produced by `webcontent_utils`.

In [ ]:
from fimbench.publish import upload_catalogntilesintos3

upload_catalogntilesintos3(
    catalog_path='out/catalog_core.json',  # unified catalog to upload
    tiles_dir='out/tiles/',                # directory of vector tiles
    bucket='sdmlab',                       # destination S3 bucket
    prefix='FIM_Database/',                # destination key prefix
)

### **Scenario 3 - publish the standardized extents to ArcGIS Online**

Needs the `publish` extra. First connect, then upsert the GeoJSON as a hosted
feature layer. Metadata fields default to FIMbench values; pass your own to
override.

In [ ]:
from fimbench import PublishFIMExtent2ArcGISOnline

gis = PublishFIMExtent2ArcGISOnline.connect_gis_oauth(
    client_id='your-agol-client-id',          # your ArcGIS Online OAuth app id
    portal_url='https://www.arcgis.com',      # portal URL (default ArcGIS Online)
    verify_cert=True,                         # verify TLS certs
)

In [ ]:
publisher = PublishFIMExtent2ArcGISOnline(
    mode_used='init',          # state label for this publisher instance
    geojson_item_id=None,      # existing uploaded GeoJSON item id (optional)
    feature_layer_item_id='',  # existing feature-layer item id (for updates)
    feature_layer_url=None,    # existing feature-layer URL (optional)
    feature_layer_title=None,  # existing feature-layer title (optional)
)

result = publisher.upsert_geojson_feature_layer(
    gis=gis,                            # the authenticated GIS connection
    geojson_path='./out/FIM_extents.geojson',  # WGS84 GeoJSON to publish
    mode='auto',                        # 'auto' | 'new' | 'update'
    published_item_id=None,             # item id to overwrite in update mode
    title=None,                         # item title (None -> FIMbench default)
    service_name=None,                  # hosted service name (optional)
    tags=None,                          # comma-separated tags (None -> default)
    summary=None,                       # short summary (None -> default)
    description=None,                   # full description (None -> default)
    folder=None,                        # AGOL folder (None -> default)
)
print(f'Successfully {result.mode_used} layer!')
print(f'Feature Layer Item ID: {result.feature_layer_item_id}')
print(f'Feature Layer URL: {result.feature_layer_url}')